In [1]:
from utils import *
from CS_feature_extractor import *
from CS_based_early_stopping import *

[nltk_data] Downloading package punkt to /home/wxr9et/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/wxr9et/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# How to run the code to get Acc and # of API calls

In [2]:
DATA_DIR = "../data/Evaluation_CoTs/sample_data/"
train_path = os.path.join(DATA_DIR, 'GSM8K_GPT4o_train.csv')
test_path = os.path.join(DATA_DIR, 'GSM8K_GPT4o_test.csv')
df_train = pd.read_csv(train_path).reset_index(drop=True)
df_test = pd.read_csv(test_path).reset_index(drop=True)

In [3]:
df_test  # Please put your question and CoT in the given format

,Name,Category,Question,Correct Answer,CoT_0,Final Answer_0,Instruction Violation_0,CoT_1,Final Answer_1,Instruction Violation_1,...,CoT_37,Final Answer_37,Instruction Violation_37,CoT_38,Final Answer_38,Instruction Violation_38,CoT_39,Final Answer_39,Instruction Violation_39,Prompt_File
0,GSM8K_test,Math,Janet’s ducks lay 16 eggs per day. She eats th...,18,Step 1: Identify the given information - Janet...,18,"[(0, 0)]",Step 1: Identify the given information - Janet...,18,"[(0, 0)]",...,Step 1: Identify the given information - Janet...,18,"[(0, 0)]",Step 1: Identify the given information - Janet...,18,"[(0, 0)]",Step 1: Identify the given information - Janet...,18,"[(0, 0)]",few_CoT.json
1,GSM8K_test,Math,A robe takes 2 bolts of blue fiber and half th...,3,Step 1: Identify the given information - The r...,3,"[(0, 0)]",Step 1: Identify the given information - The r...,3,"[(0, 0)]",...,Step 1: Identify the given information - The r...,3,"[(0, 0)]",Step 1: Identify the given information - The r...,3,"[(0, 0)]",Step 1: Identify the given information - The r...,3,"[(0, 0)]",few_CoT.json
2,GSM8K_test,Math,Josh decides to try flipping a house. He buys...,70000,Step 1: Identify the given information - Purch...,70000,"[(0, 0)]",Step 1: Identify the given information - Purch...,"$70,000","[(0, 0)]",...,Step 1: Identify the given information - Purch...,"$65,000","[(0, 0)]",Step 1: Identify the given information - Purch...,70000,"[(0, 0)]",Step 1: Identify the given information - Purch...,"$70,000","[(0, 0)]",few_CoT.json
3,GSM8K_test,Math,James decides to run 3 sprints 3 times a week....,540,Step 1: Identify the given information - James...,540 meters,"[(0, 0)]",Step 1: Identify the given information - James...,540 meters,"[(0, 0)]",...,Step 1: Identify the given information - James...,540 meters,"[(0, 0)]",Step 1: Identify the given information - James...,540 meters,"[(0, 0)]",Step 1: Identify the given information - James...,540 meters,"[(0, 0)]",few_CoT.json
4,GSM8K_test,Math,"Every day, Wendi feeds each of her chickens th...",20,Step 1: Identify the given information - Each ...,20,"[(0, 0)]",Step 1: Identify the given information - Wendi...,20,"[(0, 0)]",...,Step 1: Identify the given information - Each ...,20,"[(0, 0)]",Step 1: Identify the given information - Each ...,20,"[(0, 0)]",Step 1: Identify the given information - Wendi...,20,"[(0, 0)]",few_CoT.json
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,GSM8K_test,Math,"Out of the 200 Grade 5 students, 2/5 are boys ...",40,Step 1: Identify the total number of Grade 5 s...,40,"[(0, 0)]",Step 1: Identify the total number of students ...,40,"[(0, 0)]",...,Step 1: Identify the total number of students ...,40,"[(0, 0)]",Step 1: Identify the total number of Grade 5 s...,40,"[(0, 0)]",Step 1: Identify the total number of students ...,40,"[(0, 0)]",few_CoT.json
96,GSM8K_test,Math,Harry slept 9 hours last night. His friend Jam...,3,Step 1: Identify the given information - Harry...,3,"[(0, 0)]",Step 1: Identify the information given - Harry...,3,"[(0, 0)]",...,Step 1: Identify the given information - Harry...,3 hours,"[(0, 0)]",Step 1: Identify the given information - Harry...,3 hours,"[(0, 0)]",Step 1: Identify the amount of sleep Harry had...,3,"[(0, 0)]",few_CoT.json
97,GSM8K_test,Math,"When Freda cooks canned tomatoes into sauce, t...",12,Step 1: Identify the given information - Each ...,12,"[(0, 0)]",Step 1: Identify the given information - Each ...,12,"[(0, 0)]",...,Step 1: Identify the given information - Each ...,12,"[(0, 0)]",Step 1: Identify the given information - Each ...,12,"[(0, 0)]",Step 1: Identify the given information - Each ...,12,"[(0, 0)]",few_CoT.json
98,GSM8K_test,Math,Cars have lined up on the motorway. Some of th...,5,Step 1: Identify the given information - There...,5,"[(0, 0)]",Step 1: Identify the given information - There...,insufficient information,"[(0, 0)]",...,Step 1: Identify the given information - There...,5,"[(0

In [4]:
df_test['Model'] = 'GPT4o' # Add these lines in case you want to test multiple models to ensure fair split
df_train['Model'] = 'GPT4o'

In [5]:
feature_li = ['LEN', 'QUA_IM', 'DIF_IV', 'SIM_COT_BIGRAM', 'SIM_COT_AGG', 'SIM_AC_BIGRAM', 'SIM_AC_AGG', 'SIM_INPUT', 'STEP_COUNT',  'STEP_COHERENCE'] 
# This includes total 10 features introduced in the paper; please see extract features for more details
data_train = extract_feature(df_train,feature_li)
data_test = extract_feature(df_test,feature_li)

jaccard with bigram time cost: 5.227139472961426s
jaccard with aggregation time cost: 49.0316948890686s


100%|██████████| 100/100 [00:04<00:00, 21.25it/s]


jaccard with bigram time cost: 5.214914083480835s
jaccard with aggregation time cost: 49.12044024467468s


100%|██████████| 100/100 [00:04<00:00, 21.43it/s]


In [6]:
pd.DataFrame(data_train).head(5) # data better be saved in json format

,id,Name,Model,correct answer,CoT answers,Correctness,SIM_COT_BIGRAM,LEN,SIM_COT_AGG,SIM_AC_BIGRAM,SIM_AC_AGG,STEP_COHERENCE,SIM_INPUT,STEP_COUNT,QUA_IM,DIF_IV
0,0,GSM8K_test,GPT4o,18,"[18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.7361111111111112, 0.7692307692307692, ...","[0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, ...","[0.0, 0.7361111111111112, 0.775, 0.75, 0.65168...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.3169191919191919, 0.25520094562647755, 0.24...","[0.23170731707317072, 0.24444444444444446, 0.2...","[4, 4, 5, 4, 5, 4, 4, 5, 4, 5, 4, 4, 5, 4, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,1,GSM8K_test,GPT4o,3,"[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.8, 0.8695652173913043, 0.8888888888888...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.8, 0.7843137254901961, 0.8627450980392...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.2896825396825397, 0.3627075351213282, 0.404...","[0.26415094339622647, 0.26415094339622647, 0.2...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,2,GSM8K_test,GPT4o,70000,"[70000.0, $70,000, 70000.0, $65,000, 65000.0, ...","[1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, ...","[0.0, 0.875, 0.6981132075471699, 0.75, 0.80392...","[1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, ...","[0.0, 0.875, 0.7592592592592593, 0.71929824561...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, ...","[0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, ...","[0.38808777429467084, 0.4496891996891997, 0.32...","[0.19402985074626866, 0.2063492063492064, 0.19...","[5, 5, 4, 4, 4, 4, 7, 5, 7, 4, 4, 4, 5, 4, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,3,GSM8K_test,GPT4o,540,"[540 meters, 540 meters, 540 meters, 540 meter...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.8727272727272728, 0.8545454545454545, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, ...","[0.0, 0.8727272727272728, 0.9090909090909091, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.35998390050809403, 0.3637175739861162, 0.35...","[0.24137931034482762, 0.23728813559322037, 0.2...","[5, 7, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 5, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,4,GSM8K_test,GPT4o,20,"[20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20 ...","[1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, ...","[0.0, 0.7058823529411764, 0.7391304347826086, ...","[0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.7058823529411764, 0.72, 0.75, 0.769230...","[0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, ...","[0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, ...","[0.25706831119544593, 0.3122448392846875, 0.19...","[0.31707317073170727, 0.2790697674418605, 0.25...","[4, 5, 4, 4, 5, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [7]:
df_processed_train = pd.DataFrame(data_train)
df_processed_test = pd.DataFrame(data_test)
df_processed_test = calculate_SC_correctness(df_processed_test)

# Calculate Early Stopping Correctness with a specific window size
window_size = 5  # Define your window size
df_processed_test = calculate_ES_correctness(df_processed_test, window_size)

# Calculate Adaptive Consensus Correctness
df_processed_test = calculate_ASC_correctness(df_processed_test)

ES execution time: 0.0031 seconds
ASC execution time: 0.0215 seconds


In [8]:
df_processed_test.head() 

,id,Name,Model,correct answer,CoT answers,Correctness,SIM_COT_BIGRAM,LEN,SIM_COT_AGG,SIM_AC_BIGRAM,...,STEP_COHERENCE,SIM_INPUT,STEP_COUNT,QUA_IM,DIF_IV,SC_correctness,ES_correctness,ES_steps,asc_correctness,asc_steps
0,0,GSM8K_test,GPT4o,18,"[18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.7361111111111112, 0.7692307692307692, ...","[0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, ...","[0.0, 0.7361111111111112, 0.775, 0.75, 0.65168...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",...,"[0.3169191919191919, 0.25520094562647755, 0.24...","[0.23170731707317072, 0.24444444444444446, 0.2...","[4, 4, 5, 4, 5, 4, 4, 5, 4, 5, 4, 4, 5, 4, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,1,5,1,4
1,1,GSM8K_test,GPT4o,3,"[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.8, 0.8695652173913043, 0.8888888888888...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.8, 0.7843137254901961, 0.8627450980392...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",...,"[0.2896825396825397, 0.3627075351213282, 0.404...","[0.26415094339622647, 0.26415094339622647, 0.2...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,1,5,1,4
2,2,GSM8K_test,GPT4o,70000,"[70000.0, $70,000, 70000.0, $65,000, 65000.0, ...","[1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, ...","[0.0, 0.875, 0.6981132075471699, 0.75, 0.80392...","[1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, ...","[0.0, 0.875, 0.7592592592592593, 0.71929824561...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, ...",...,"[0.38808777429467084, 0.4496891996891997, 0.32...","[0.19402985074626866, 0.2063492063492064, 0.19...","[5, 5, 4, 4, 4, 4, 7, 5, 7, 4, 4, 4, 5, 4, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,1,40,1,40
3,3,GSM8K_test,GPT4o,540,"[540 meters, 540 meters, 540 meters, 540 meter...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.8727272727272728, 0.8545454545454545, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, ...","[0.0, 0.8727272727272728, 0.9090909090909091, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",...,"[0.35998390050809403, 0.3637175739861162, 0.35...","[0.24137931034482762, 0.23728813559322037, 0.2...","[5, 7, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 5, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,5,0,4
4,4,GSM8K_test,GPT4o,20,"[20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20 ...","[1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, ...","[0.0, 0.7058823529411764, 0.7391304347826086, ...","[0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.7058823529411764, 0.72, 0.75, 0.769230...","[0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, ...",...,"[0.25706831119544593, 0.3122448392846875, 0.19...","[0.31707317073170727, 0.2790697674418605, 0.25...","[4, 5, 4, 4, 5, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,1,5,1,4


In [9]:
# TO DO 
# Write doc on how to run the python file. (special explanations for feature extraction)

In [10]:
feature_li = ['LEN', 'SIM_COT_BIGRAM', 'SIM_COT_AGG', 'SIM_AC_BIGRAM', 'SIM_AC_AGG', 'SIM_INPUT', 'STEP_COUNT',  'STEP_COHERENCE']  # We can take less if for test
df_confidence_scores,coefs = trained_LR_model(df_processed_train, feature_li, report_auroc=False,train_mode = True) # Note that we first try dataset to get the coefficient/ You can also analyze the train data performance since it's also splitted to train and validation if needed

Optimization terminated successfully.
         Current function value: 0.500708
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            Correctness   No. Observations:                 2800
Model:                          Logit   Df Residuals:                     2791
Method:                           MLE   Df Model:                            8
Date:                Tue, 04 Feb 2025   Pseudo R-squ.:                  0.1313
Time:                        21:43:47   Log-Likelihood:                -1402.0
converged:                       True   LL-Null:                       -1613.9
Covariance Type:            nonrobust   LLR p-value:                 1.488e-86
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.5902      0.430     -1.373      0.170      -1.433       0.252
LEN              

In [11]:
coefs

array([-0.59023011, -0.17887917, -2.47526597,  2.57520725,  0.68997781,
        1.65216567, -2.61836719, -0.04469021,  3.54958297])

In [12]:
df_confidence_scores = customized_LR_model(df_processed_test, feature_li, coefs[1:], coefs[0],report_auroc=False)

In [13]:
N = 4
threshold = 0.5

# Applying early stopping mechanism on sample data
df_final = CS_early_stopping(df=df_confidence_scores, threshold=threshold, N=N)

SC_ACC : 0.75
ES_ACC : 0.76
CS_ACC : 0.74
SC_Avg_Steps : 40
ES_Avg_Steps : 9.51
CS_Avg_Steps : 5.54
ASC_Avg_Steps : 8.42
ASC_ACC : 0.74


In [15]:
# Get best steps for all rows
def get_best_step(row):
    # Get only the first CS_steps values from the confidence_score list
    scores = row['confidence_score'][:row['CS_steps']]
    # Return the argmax (index of maximum value)
    return np.argmax(scores)

df_confidence_scores['best_step'] = df_confidence_scores.apply(get_best_step, axis=1)

# Create the column names for each row
cot_cols = [f'CoT_{step}' for step in df_confidence_scores['best_step']]
final_answer_cols = [f'Final Answer_{step}' for step in df_confidence_scores['best_step']]

# Create result DataFrame with best values
result_df = pd.DataFrame({
    'best_CoT': [df_test.loc[i, col] for i, col in enumerate(cot_cols)],
    'best_Final_Answer': [df_test.loc[i, col] for i, col in enumerate(final_answer_cols)]
})

In [16]:
result_df ## Best Rationale Selection

,best_CoT,best_Final_Answer
0,Step 1: Identify the given information - Janet...,18
1,Step 1: Identify the given information - The r...,3
2,Step 1: Identify the given information - Purch...,70000
3,Step 1: Identify the given information - James...,540 meters
4,Step 1: Identify the given information - Wendi...,20
...,...,...
95,Step 1: Identify the total number of students ...,40
96,Step 1: Identify the given information - Harry...,3
97,Step 1: Identify the given information - Each ...,12
98,Step 1: Identify the given information - There...,5
